# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택, 필터링, 정렬하고 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 제출 정보
- 이름: 박성주
- GitHub ID: mikeypark
- 작성일: 2026-09-22


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소, 환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [2]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)

Python 실행 파일: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\.venv\Scripts\python.exe
현재 작업 폴더: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\chapter04
프로젝트 루트: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study
데이터 폴더: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\data\raw
결과 저장 폴더: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [3]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')

데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [4]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


**[결과 관찰]**

- customers: (150, 6) - customer_id, name, gender, age, city, signup_date
- products: (100, 4) - product_id, product_name, category, price
- orders: (300, 5) - order_id, customer_id, order_date, payment_method, order_status
- order_items: (764, 5) - order_item_id, order_id, product_id, quantity, unit_price

필수 컬럼은 모두 있어서 KeyError 없이 통과됐다. customers에 `name` 컬럼이 있기 때문에 출력할 때 이름이 노출되지 않도록 주의해야 한다.


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [5]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))

,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


**[결과 관찰]**

도시별 고객 수: 성남 21명, 광주 17명, 부산 16명, 대구 15명, 서울 15명 순으로 성남이 가장 많다.
서울, 부산 합산 31명(150명 중 약 21%).

**[나의 해석과 판단]**

필터링 전에 `value_counts()`로 도시명 표기를 먼저 확인했다. 예를 들어 `'서울'` 대신 다른 표기가 쓰여 있으면 필터 결과가 0건으로 나오기 때문이다.

**[분석 의미]**

성남, 광주 같은 수도권 인접 도시에 고객이 많이 분포되어 있어 지역별 마케팅이나 배송 전략을 세울 때 참고할 수 있다.

**[한계]**

샘플 데이터라서 실제 고객 분포와 다를 수 있고, 도시명 표기 방식이 실데이터와 다를 경우 적용 전에 한번 더 확인이 필요하다.


## 5. 정렬과 주문 상태 확인


In [6]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-06-04
8,9,송지민,M,69,서울,2025-10-19
54,55,윤영철,M,69,광주,2023-07-31
78,79,이서준,M,69,부산,2023-07-14
136,137,김선영,M,68,울산,2024-11-28
45,46,김서현,M,67,대전,2026-05-13
132,133,김성진,M,67,성남,2026-05-29
123,124,김시우,M,67,대구,2024-09-24
11,12,김정남,F,66,대전,2024-09-30
57,58,김성훈,F,66,성남,2025-06-14


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

**[결과 관찰]**

주문 상태: completed 184건, cancelled 64건, refunded 52건 (총 300건).
완료 주문 비율 약 61.3%, 취소와 환불을 합치면 38.7%가 비완료 주문이다.

**[나의 해석과 판단]**

취소, 환불 비율이 꽤 높다. 전체 주문 기준으로 집계하면 실제보다 금액이 크게 나오기 때문에 완료 주문만 따로 필터링해야 한다.


## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [7]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())

전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


**[결과 관찰]**

전체 주문 상세 764행의 line_total 합계: 255,610,000원.
이 시점에는 취소, 환불 주문이 포함되어 있어 완료 주문 매출로 볼 수 없다.

**[나의 해석과 판단]**

`line_total = quantity × unit_price` 계산은 단순하지만, 이 합계를 "매출"이라고 부르려면 주문 상태 필터링이 먼저 되어야 한다.


## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [8]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')

orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

**[결과 관찰]**

- orders.order_id 중복 수: 0
- 병합 전 행 수: 764 / 병합 후 행 수: 764 (행 수 동일)
- _merge: both 764건, left_only 0건 (미매칭 없음)

**[나의 해석과 판단]**

`validate='many_to_one'`이 통과됐으므로 orders.order_id가 실제로 중복 없이 유일하다는 게 확인됐다. left_only가 없어서 모든 주문 상세에 대응하는 주문 헤더가 있다. 병합 전후 행 수가 같아서 행이 늘거나 줄지 않았다.

**[분석 의미]**

validate 없이 merge하면 orders에 중복 order_id가 있을 때 행 수가 급격하게 늘어나 집계 금액이 실제보다 훨씬 커질 수 있다. indicator로 left_only를 확인하지 않으면 상태 정보가 없는 행이 집계에 섞여 들어갈 수 있다.

**[한계]**

샘플 데이터는 깔끔하게 만들어져 있지만 실데이터에는 orders에 매칭되지 않는 order_item이 있을 수 있다. 그런 경우 제외할지 별도로 표시할지 미리 정해둬야 한다.


## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [9]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())

날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


**[결과 관찰]**

- 날짜 변환 실패: 0건
- 전체 주문 상세 행 수: 764 → 완료 주문 상세 행 수: 474
- 완료 주문 line_total 합계: 148,990,000원

**[나의 해석과 판단]**

전체 764행 중 474행(62%)이 completed 주문이다. 취소, 환불 금액 106,620,000원을 제외한 148,990,000원을 이번 분석의 기준 금액으로 사용한다. 반품 처리 시점이나 할인 적용 여부는 이 데이터에 포함되지 않아서 회계상 순매출이라고 단정하지 않는다.


## 9. 상품 데이터 병합과 검증


In [10]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')

products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

**[결과 관찰]**

- products.product_id 중복 수: 0
- 병합 전: 474 / 병합 후: 474 (행 수 동일)
- _merge: both 474건, left_only 0건

**[나의 해석과 판단]**

두 번째 merge도 validate와 indicator로 안전성을 확인했다. 미매칭 상품이 없으므로 category, product_name 없이 집계되는 행이 없다. 만약 left_only가 있었다면 category가 NaN인 채로 groupby에 들어가 NaN 카테고리 행이 생겼을 것이다.


## 10. 카테고리별, 상품별 매출


In [11]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))

,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


**[결과 관찰]**

카테고리별 완료 주문 매출 (단위: 원):
1. 스포츠: 31,743,000 (21.3%)
2. 전자기기: 26,400,000 (17.7%)
3. 생활용품: 23,915,000 (16.1%)
4. 뷰티: 23,383,000 (15.7%)
5. 식품: 16,573,000 (11.1%)
6. 도서: 16,389,000 (11.0%)
7. 패션: 10,587,000 (7.1%)

매출 1위 상품: 스포츠 상품 041

**[나의 해석과 판단]**

스포츠가 전체의 21%로 1위다. 수량도 295개로 가장 많아서 단가와 판매량이 모두 높은 카테고리다. 패션은 매출과 수량 모두 꼴찌라서 마케팅이나 재고 전략을 다시 검토해볼 필요가 있어 보인다.

**[분석 의미]**

스포츠와 전자기기 상위 2개가 전체의 39%를 차지한다. 이 두 카테고리를 중심으로 재고나 프로모션을 집중하면 효율적일 것 같다.

**[한계]**

`total_sales`는 completed 주문의 line_total 합계이고 할인이나 배송비는 반영되지 않아 순매출과는 다르다. 카테고리별로 상품 수가 다르기도 해서(스포츠 19개, 패션 11개) 단순 비교만으로 판단하기는 어렵다.


## 11. 월별 매출


In [12]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)

,order_month,total_sales,order_count,average_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


**[결과 관찰]**

2025-07 ~ 2026-07 총 13개월. 최고 매출 월: 2025-10 (25,766,000원, 26건). 최저 매출 월: 2026-07 (2,188,000원, 2건이라 참고 수준). 월별 합계: 148,990,000원으로 완료 주문 전체 합과 일치한다.

**[나의 해석과 판단]**

2025년 10월에 매출이 집중된다. 2026년 상반기는 비슷한 수준이지만 6, 7월은 데이터가 너무 적어서 트렌드를 판단하기 어렵다. order_count는 `nunique`로 계산해서 같은 주문이 두 번 세이지 않게 했다.

**[분석 의미]**

10월 매출이 유독 높은 이유를 파악하면 다음 해 같은 시기에 활용할 수 있다. 프로모션이나 시즌 효과인지 확인해볼 만하다.


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [13]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))

,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


**[결과 관찰]**

최고 구매 고객: Customer 117 (성남, 5회 주문, 4,100,000원). 고객별 합계: 148,990,000원으로 완료 주문 전체 합과 일치한다. 상위 5명 중 서울 고객 2명.

**[나의 해석과 판단]**

이름 대신 customer_label을 사용해서 출력에 개인 이름이 나오지 않게 했다. validate='one_to_one'으로 고객 ID가 집계 후에도 중복 없이 유일한지 확인했다. 합계도 전체와 일치해서 누락 없이 집계됐다.

**[분석 의미]**

구매 금액이 높은 고객을 파악해두면 이후 우수 고객 관리나 맞춤 혜택 설계에 활용할 수 있다.


## 13. 결과 저장하기


In [14]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)

ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 442
ch04_customer_sales.csv True 3387


## 14. LLM 코드 검증 연습

LLM에게 다음 프롬프트를 입력했습니다:

```text
orders와 order_items를 사용해
completed 주문 기준 월별 금액을 계산하려고 합니다.

다음을 반드시 포함해 주세요.
- line_total = quantity × unit_price
- orders.order_id 고유성 확인
- many-to-one merge + indicator
- 병합 전후 행 수 확인
- 미매칭 확인
- 날짜 변환 실패 확인
- completed만 필터링
- 월별 total_sales와 고유 order_count 계산
- 결과를 회계상 순매출이라고 단정하지 않기
```

**[LLM 제안 코드 - 검증 대상]**

```python
category_sales = order_items.groupby('category')['line_total'].sum()
```

**[검증 결과]**

| 항목 | 판정 | 비고 |
|------|------|------|
| 실제 컬럼명 사용 | ❌ | `order_items`에 `category` 컬럼 없음, products와 merge 필요 |
| 주문 상태 필터링 | ❌ | completed 필터 없음, 취소, 환불 포함 집계됨 |
| 병합 검증 | ❌ | validate, indicator, 행 수 확인 없음 |
| 날짜 변환 확인 | ❌ | 해당 없음 (월별이 아닌 카테고리별) |
| 결과 의미 과장 방지 | ❌ | 주석, 설명 없음 |

**[나의 해석과 판단]**

LLM이 제안한 코드는 `order_items`에 `category` 컬럼이 없어서 바로 실행하면 `KeyError`가 난다. 설령 실행이 된다 해도 completed 필터가 없어서 취소, 환불 금액이 그대로 합산된다. LLM 코드는 실행 여부보다 데이터 구조, 계산 범위, 검증 항목 포함 여부를 기준으로 판단해야 한다.


## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [15]:
# 과제 1. 40세 이상 고객 추출
customers_over_40 = customers[customers['age'] >= 40]
print('40세 이상 고객 수:', len(customers_over_40))
display(customers_over_40[['customer_id', 'gender', 'age', 'city']].head())

40세 이상 고객 수: 78


,customer_id,gender,age,city
2,3,F,61,성남
3,4,F,55,울산
6,7,F,53,인천
8,9,M,69,서울
9,10,F,62,울산


In [16]:
# 과제 2. 상품 가격 낮은 순 10개
display(
    products.sort_values('price', ascending=True)
    .head(10)
    [['product_name', 'category', 'price']]
)

,product_name,category,price
5,전자기기 상품 006,전자기기,5000
27,뷰티 상품 028,뷰티,5000
97,스포츠 상품 098,스포츠,10000
46,생활용품 상품 047,생활용품,11000
94,전자기기 상품 095,전자기기,20000
49,뷰티 상품 050,뷰티,23000
82,전자기기 상품 083,전자기기,24000
86,도서 상품 087,도서,25000
41,패션 상품 042,패션,28000
58,뷰티 상품 059,뷰티,32000


In [17]:
# 과제 3. 결제수단별 주문 수와 비율
pm_counts = orders['payment_method'].value_counts()
pm_ratio = (pm_counts / pm_counts.sum() * 100).round(2)
pm_summary = pd.DataFrame({'count': pm_counts, 'ratio(%)': pm_ratio})
display(pm_summary)

,count,ratio(%)
payment_method,,
kakao_pay,79,26.33
naver_pay,77,25.67
bank_transfer,74,24.67
card,70,23.33


In [18]:
# 과제 4. 카테고리별 평균 상품 가격
category_avg_price = (
    products.groupby('category')['price']
    .mean()
    .round(0)
    .sort_values(ascending=False)
    .rename('avg_price')
    .reset_index()
)
display(category_avg_price)

,category,avg_price
0,식품,137143.0
1,뷰티,117688.0
2,패션,115909.0
3,스포츠,111579.0
4,도서,106857.0
5,전자기기,101588.0
6,생활용품,96438.0


In [19]:
# 과제 5. 완료 주문과 전체 주문 금액 차이
total_all = order_items['line_total'].sum()
total_completed = completed_order_sales['line_total'].sum()
diff = total_all - total_completed

print(f'전체 주문 금액:   {total_all:,.0f}원')
print(f'완료 주문 금액:   {total_completed:,.0f}원')
print(f'차이(취소+환불): {diff:,.0f}원')
print(f'비완료 비율:      {diff/total_all*100:.1f}%')

전체 주문 금액:   255,610,000원
완료 주문 금액:   148,990,000원
차이(취소+환불): 106,620,000원
비완료 비율:      41.7%


**과제 6. 병합 검증 포함 고객별 구매 금액 요청 프롬프트**

```text
orders, order_items, customers 테이블을 사용해
완료 주문(order_status == 'completed') 기준 고객별 구매 금액을 계산해 주세요.

컬럼 정보:
- order_items: order_item_id, order_id, product_id, quantity, unit_price
- orders: order_id, customer_id, order_date, payment_method, order_status
- customers: customer_id, name, gender, age, city, signup_date

반드시 포함할 사항:
1. line_total = quantity × unit_price 파생 컬럼 생성
2. orders.order_id 중복 여부 확인
3. order_items와 orders 병합 시 validate='many_to_one', indicator=True 사용
4. 병합 전후 행 수 출력 및 _merge 값 확인
5. order_status == 'completed' 필터링 후 집계
6. 고객별 order_id.nunique()로 주문 수 계산
7. customers와 병합 시 validate='one_to_one' 사용, city만 포함
8. 개인 이름 컬럼 제외, customer_label로 익명화
9. 결과를 회계상 순매출로 단정하지 말 것
```


## 16. 총합 일치 검증 (Evidence)


In [20]:
base = completed_order_sales['line_total'].sum()
cat_sum = category_sales['total_sales'].sum()
monthly_sum = monthly_summary['total_sales'].sum()
cust_sum = customer_sales['total_sales'].sum()

print(f'완료 주문 상세 합계:  {base:,.0f}')
print(f'카테고리별 합계:      {cat_sum:,.0f}  PASS={base==cat_sum}')
print(f'월별 합계:            {monthly_sum:,.0f}  PASS={base==monthly_sum}')
print(f'고객별 합계:          {cust_sum:,.0f}  PASS={base==cust_sum}')

완료 주문 상세 합계:  148,990,000
카테고리별 합계:      148,990,000  PASS=True
월별 합계:            148,990,000  PASS=True
고객별 합계:          148,990,000  PASS=True


**[Chapter 04 Evidence 요약]**

```
1. Notebook 실행: 완료

2. 병합 검증
   orders.order_id 중복: 0
   orders merge 전/후 행 수: 764 / 764
   orders merge 미매칭: 0
   products merge 전/후 행 수: 474 / 474
   products merge 미매칭: 0

3. 계산 범위
   사용 주문 상태: completed
   line_total: quantity × unit_price
   날짜 변환 실패: 0

4. 교차 검증
   완료 주문 line_total 합계: 148,990,000
   category_sales 합계:       148,990,000  PASS
   monthly_summary 합계:      148,990,000  PASS
   customer_sales 합계:       148,990,000  PASS

5. 저장 파일
   ch04_category_sales.csv: 존재
   ch04_product_sales.csv:  존재
   ch04_monthly_sales.csv:  존재
   ch04_customer_sales.csv: 존재

6. LLM 검증
   LLM이 빠뜨린 항목: category 컬럼 위치 오류, completed 필터 누락,
   validate/indicator/행 수 확인 없음
```


## Chapter 04 최종 인사이트

### 가장 의미 있다고 생각한 결과 2가지

1. **완료 vs 전체 금액 차이**: 전체 255,610,000원 중 완료 주문은 148,990,000원(58.3%)으로 취소, 환불이 전체의 41.7%를 차지한다. "매출"을 어느 기준으로 집계하느냐에 따라 금액이 1.7배 차이난다.

2. **스포츠 카테고리 1위**: 스포츠(21.3%)가 전자기기(17.7%)를 앞서며 매출 1위다. 수량 기준으로도 1위여서 단가와 판매량이 모두 높은 카테고리다.

### 그 결과를 뒷받침하는 수치/표

- 전체: 255,610,000원 / 완료: 148,990,000원 / 차이: 106,620,000원
- 스포츠 total_sales: 31,743,000원, total_quantity: 295개

### 추가로 확인하고 싶은 질문

- 카테고리별 취소, 환불 비율은 다른가? (특정 카테고리가 환불이 많은지)
- 결제수단별 completed 비율 차이가 있는가?
- 상위 구매 고객의 재구매 주기는 얼마나 되는가?

### 현재 결과의 한계

- 샘플 데이터(150명, 300주문)로 실제 서비스 패턴과 다를 수 있다.
- `total_sales`는 할인, 쿠폰, 배송비가 반영되지 않아 회계상 순매출과 다르다.
- 2026-06, 07 데이터가 적어서 최근 트렌드 해석에 한계가 있다.
